# MIA Project 2 — Shunt Valve Artifact Detection & Catheter Length Estimation

**Tasks:**
- **Task A:** Segment shunt valve artifact (3D U-Net, 1-channel input)
- **Task B:** Segment catheter outside the artifact (3D U-Net, 2-channel input: MRI + predicted artifact)
- **Task C:** Estimate total catheter length in mm (skeleton → spline → arc length + artifact gap)

**Runs in both Colab and locally** — see the environment cell below.

## 1. Setup

In [ ]:
# Install dependencies (skip if already installed locally — only needed first run)
!pip install -q nibabel SimpleITK scikit-image scipy torch torchvision tqdm pandas matplotlib
!pip install -q torchio

In [ ]:
import os, json, warnings, itertools, hashlib
from pathlib import Path
from typing import Tuple, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import nibabel as nib
import SimpleITK as sitk
import torchio as tio

from scipy.ndimage import label, binary_fill_holes, binary_erosion, binary_dilation, convolve
from scipy.interpolate import splprep, splev
from scipy.spatial import cKDTree

from skimage.morphology import skeletonize, ball
from skimage.exposure import equalize_adapthist

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
print("Imports OK")

In [ ]:
# ─── Environment detection: Colab vs local ────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/MIAProject2")
    CACHE_DIR    = Path("/content/preproc_cache")
    IN_COLAB = True
except ImportError:
    # Local: set MIA_PROJECT_ROOT env var, or edit this default.
    # Expected layout under PROJECT_ROOT:
    #   data/Train/subjXXX/...
    #   data/Test/subjXXX/...
    #   output/   (will be created)
    PROJECT_ROOT = Path(os.environ.get("MIA_PROJECT_ROOT", "."))
    CACHE_DIR    = PROJECT_ROOT / "preproc_cache"
    IN_COLAB = False

DATA_ROOT = PROJECT_ROOT / "data" / "Train"
TEST_ROOT = PROJECT_ROOT / "data" / "Test"
OUT_DIR   = PROJECT_ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ─── Device selection ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─── Training config ──────────────────────────────────────────────────────────
# n_workers: A100 + High-RAM Colab handles 8. Drop to 2 on a laptop.
CFG = dict(
    patch_size = (96, 96, 96),
    batch_size = 4,
    lr         = 3e-4,
    epochs_a   = 60,
    epochs_b   = 60,
    val_frac   = 0.15,
    n_workers  = 8 if IN_COLAB else 2,
)

print(f"Environment : {'Colab' if IN_COLAB else 'Local'}")
print(f"Device      : {DEVICE}")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Output dir  : {OUT_DIR.resolve()}")

## 2. Data utilities

In [5]:
def load_nifti(path: Path) -> Tuple[np.ndarray, np.ndarray, nib.Nifti1Image]:
    """Return (data_float32, voxel_spacing_mm, nibabel_img)."""
    img = nib.load(str(path))
    data = img.get_fdata(dtype=np.float32)
    spacing = np.array(img.header.get_zooms()[:3], dtype=np.float32)
    return data, spacing, img


def save_nifti(mask: np.ndarray, ref_img: nib.Nifti1Image, path: Path):
    """Save uint8 binary mask using the reference image's affine and header."""
    out = nib.Nifti1Image(mask.astype(np.uint8), ref_img.affine, ref_img.header)
    nib.save(out, str(path))


def get_subject_paths(root: Path, labeled: bool = True) -> List[dict]:
    """Return sorted list of per-subject path dicts."""
    subjects = sorted(root.glob("subj*"))
    records = []
    for s in subjects:
        sid = s.name
        rec = {"id": sid, "image": s / f"{sid}_image.nii.gz"}
        if labeled:
            rec["artifact"] = s / f"{sid}_artifact.nii.gz"
            rec["catheter"] = s / f"{sid}_catheter.nii.gz"
            json_path = s / f"{sid}.json"
            if json_path.exists():
                rec["json"] = json_path
        records.append(rec)
    return records


# Sanity check
if DATA_ROOT.exists():
    records = get_subject_paths(DATA_ROOT)
    r0 = records[0]
    img, sp, _ = load_nifti(r0["image"])
    print(f"Subjects: {len(records)} | Volume shape: {img.shape} | Spacing: {sp} mm")
else:
    print(f"DATA_ROOT '{DATA_ROOT}' not found — check your project root.")

DATA_ROOT '/content/drive/MyDrive/Colab Notebooks/MIAProject2/data/Train' not found — check your project root.


In [ ]:
def clahe_volume(vol: np.ndarray) -> np.ndarray:
    """Apply 2D CLAHE slice-by-slice along the axial axis."""
    out = np.zeros_like(vol)
    v_min, v_max = vol.min(), vol.max()
    vol_norm = (vol - v_min) / (v_max - v_min + 1e-8)
    for z in range(vol.shape[2]):
        out[..., z] = equalize_adapthist(vol_norm[..., z], clip_limit=0.03)
    return out


def intensity_normalise(vol: np.ndarray) -> np.ndarray:
    """Clip to [0.5%, 99.5%] percentile then z-score."""
    lo, hi = np.percentile(vol, 0.5), np.percentile(vol, 99.5)
    vol = np.clip(vol, lo, hi)
    return (vol - vol.mean()) / (vol.std() + 1e-8)


def n4_correct(vol: np.ndarray) -> np.ndarray:
    """N4 with shrink-factor downsampling for speed (~30s vs 2-3 min/vol)."""
    img = sitk.GetImageFromArray(vol.transpose(2, 1, 0))   # XYZ → ZYX
    img = sitk.Cast(img, sitk.sitkFloat32)
    mask = sitk.OtsuThreshold(img, 0, 1, 200)
    shrink = 4
    img_s = sitk.Shrink(img, [shrink] * img.GetDimension())
    mask_s = sitk.Shrink(mask, [shrink] * img.GetDimension())
    n4 = sitk.N4BiasFieldCorrectionImageFilter()
    n4.SetMaximumNumberOfIterations([50, 50, 30, 20])
    _ = n4.Execute(img_s, mask_s)
    log_bias_full = n4.GetLogBiasFieldAsImage(img)
    corr = sitk.Exp(sitk.Log(img + 1e-6) - log_bias_full)
    return sitk.GetArrayFromImage(corr).transpose(2, 1, 0).astype(np.float32)


def preprocess_volume(path: Path, use_cache: bool = True,
                      n4: bool = False) -> np.ndarray:
    """
    Preprocess: load → optional N4 → CLAHE → z-score.
    N4 cache uses _n4_pre.npy suffix; non-N4 cache uses _pre.npy (untouched).
    """
    cache_key = Path(path).name.replace('.nii.gz', '').replace('.nii', '')
    suffix = '_n4_pre.npy' if n4 else '_pre.npy'
    cache_path = CACHE_DIR / (cache_key + suffix)
    if use_cache and cache_path.exists():
        return np.load(str(cache_path))

    vol, _, _ = load_nifti(path)
    if n4:
        vol = n4_correct(vol)
    vol = clahe_volume(vol)
    vol = intensity_normalise(vol).astype(np.float32)

    if use_cache:
        np.save(str(cache_path), vol)
    return vol

## 3. Model & dataset

In [7]:
class ConvBnRelu(nn.Module):
    """Two 3x3x3 convolutions with InstanceNorm and LeakyReLU."""
    def __init__(self, cin, cout, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(cin, cout, k, padding=p, bias=False),
            nn.InstanceNorm3d(cout),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(cout, cout, k, padding=p, bias=False),
            nn.InstanceNorm3d(cout),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = ConvBnRelu(cin, cout)
    def forward(self, x):
        return self.conv(self.pool(x))


class UpBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.up = nn.ConvTranspose3d(cin, cout, kernel_size=2, stride=2)
        self.conv = ConvBnRelu(cin, cout)
    def forward(self, x, skip):
        x = self.up(x)
        diff = [s - x.shape[i+2] for i, s in enumerate(skip.shape[2:])]
        x = F.pad(x, [0, diff[2], 0, diff[1], 0, diff[0]])
        return self.conv(torch.cat([skip, x], dim=1))


class UNet3D(nn.Module):
    """
    4-level 3D U-Net.
    Task A: in_channels=1 (MRI only)
    Task B: in_channels=2 (MRI + artifact mask)
    """
    def __init__(self, in_channels: int = 1, base: int = 16, out_classes: int = 1):
        super().__init__()
        b = base
        self.enc0 = ConvBnRelu(in_channels, b)
        self.enc1 = DownBlock(b, b*2)
        self.enc2 = DownBlock(b*2, b*4)
        self.enc3 = DownBlock(b*4, b*8)
        self.bot  = DownBlock(b*8, b*16)
        self.dec3 = UpBlock(b*16, b*8)
        self.dec2 = UpBlock(b*8, b*4)
        self.dec1 = UpBlock(b*4, b*2)
        self.dec0 = UpBlock(b*2, b)
        self.head = nn.Conv3d(b, out_classes, 1)

    def forward(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b  = self.bot(e3)
        d3 = self.dec3(b, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        d0 = self.dec0(d1, e0)
        return self.head(d0)  # raw logits, shape (B, 1, X, Y, Z)


_net_a = UNet3D(in_channels=1, base=16)
_net_b = UNet3D(in_channels=2, base=16)
print(f"Task-A U-Net params: {sum(p.numel() for p in _net_a.parameters())/1e6:.2f}M")
print(f"Task-B U-Net params: {sum(p.numel() for p in _net_b.parameters())/1e6:.2f}M")
del _net_a, _net_b

Task-A U-Net params: 5.64M
Task-B U-Net params: 5.64M


In [ ]:
class PatchDataset(Dataset):
    """
    Random patch sampling from preprocessed volumes.
    Task A: input = MRI (1ch), label = artifact mask
    Task B: input = MRI + artifact mask (2ch), label = catheter mask

    train=True : random origins + TorchIO augmentation (image+labels stay aligned).
    train=False: deterministic origins (per-subject MD5-seeded RNG), no augmentation.
    """
    def __init__(self, records: List[dict], task: str = 'A',
                 patch_size: Tuple = (96, 96, 96),
                 patches_per_vol: int = 16,
                 fg_oversample: float = 0.5,
                 train: bool = True):
        assert task in ('A', 'B')
        self.records = records
        self.task = task
        self.patch_size = np.array(patch_size)
        self.patches_per_vol = patches_per_vol
        self.fg_oversample = fg_oversample
        self.train = train
        self._cache = {}

        if train:
            self.aug = tio.Compose([
                tio.RandomFlip(axes=(0, 1, 2), flip_probability=0.5),
                tio.RandomAffine(scales=(0.9, 1.1), degrees=15,
                                 isotropic=False, p=0.5),
                tio.RandomElasticDeformation(num_control_points=7,
                                             max_displacement=4, p=0.3),
                tio.RandomBiasField(coefficients=0.3, order=3, p=0.3),
                tio.RandomGamma(log_gamma=(-0.3, 0.3), p=0.3),
            ])
            self.det_origins = None
        else:
            self.aug = None
            self.det_origins = self._precompute_origins()

    def _precompute_origins(self):
        """Reproducible patch origins per subject, biased toward foreground.

        Uses MD5 of the subject ID (stable across processes — unlike Python's
        built-in hash which depends on PYTHONHASHSEED).
        """
        origins = {}
        for rec in self.records:
            sid = rec["id"]
            seed = int.from_bytes(hashlib.md5(sid.encode()).digest()[:8], 'big')
            rng = np.random.default_rng(seed)
            vol, art, cat = self._load(rec)
            label_vol = art if self.task == 'A' else cat
            sh = np.array(vol.shape)
            ps = self.patch_size
            fg = np.argwhere(label_vol > 0)

            sub_origins = []
            for _ in range(self.patches_per_vol):
                if len(fg) > 0 and rng.random() < self.fg_oversample:
                    centre = fg[rng.integers(len(fg))]
                else:
                    centre = np.array([
                        rng.integers(p // 2, max(p // 2 + 1, s - p // 2))
                        for s, p in zip(sh, ps)
                    ])
                lo = np.clip(centre - ps // 2, 0, sh - ps)
                sub_origins.append(tuple(int(l) for l in lo))
            origins[sid] = sub_origins
        return origins

    def _load(self, rec: dict):
        sid = rec["id"]
        if sid not in self._cache:
            vol = preprocess_volume(rec["image"])
            art, _, _ = load_nifti(rec["artifact"])
            cat, _, _ = load_nifti(rec["catheter"])
            self._cache[sid] = (vol.astype(np.float32),
                                art.astype(np.float32),
                                cat.astype(np.float32))
        return self._cache[sid]

    def _sample_origin(self, vol_shape, label_vol):
        """Pick random patch origin, biased toward foreground voxels."""
        ps = self.patch_size
        sh = np.array(vol_shape)
        fg = np.argwhere(label_vol > 0)

        if len(fg) > 0 and np.random.rand() < self.fg_oversample:
            centre = fg[np.random.randint(len(fg))]
        else:
            centre = np.array([np.random.randint(p // 2, max(p // 2 + 1, s - p // 2))
                               for s, p in zip(sh, ps)])

        lo = np.clip(centre - ps // 2, 0, sh - ps)
        return tuple(int(l) for l in lo)

    def _extract(self, vol, lo):
        ps = self.patch_size
        return vol[lo[0]:lo[0]+ps[0], lo[1]:lo[1]+ps[1], lo[2]:lo[2]+ps[2]]

    def _origin_for(self, rec, vol_shape, label_vol, k_in_subject):
        if self.train:
            return self._sample_origin(vol_shape, label_vol)
        return self.det_origins[rec["id"]][k_in_subject]

    def _augment(self, img_p, lbl_p, extra_p=None):
        """TorchIO transforms — geometric ops apply identically to image and
        labels (LabelMap uses nearest-neighbour); intensity ops skip labels."""
        sub_dict = {
            "image": tio.ScalarImage(tensor=torch.from_numpy(img_p[None]).float()),
            "label": tio.LabelMap(tensor=torch.from_numpy(lbl_p[None]).float()),
        }
        if extra_p is not None:
            sub_dict["extra"] = tio.LabelMap(
                tensor=torch.from_numpy(extra_p[None]).float())
        out = self.aug(tio.Subject(**sub_dict))
        img_a = out["image"].data[0].numpy()
        lbl_a = out["label"].data[0].numpy()
        if extra_p is not None:
            extra_a = out["extra"].data[0].numpy()
            return img_a, lbl_a, extra_a
        return img_a, lbl_a

    def __len__(self):
        return len(self.records) * self.patches_per_vol

    def __getitem__(self, idx):
        rec_idx = idx // self.patches_per_vol
        k = idx % self.patches_per_vol
        rec = self.records[rec_idx]
        vol, art, cat = self._load(rec)

        if self.task == 'A':
            lo = self._origin_for(rec, vol.shape, art, k)
            img_p = self._extract(vol, lo)
            lbl_p = self._extract(art, lo)

            if self.train:
                img_p, lbl_p = self._augment(img_p, lbl_p)

            img_t = torch.from_numpy(np.ascontiguousarray(img_p)).unsqueeze(0).float()
            lbl_t = torch.from_numpy(np.ascontiguousarray(lbl_p)).unsqueeze(0).float()

        else:  # Task B
            lo = self._origin_for(rec, vol.shape, cat, k)
            img_p = self._extract(vol, lo)
            art_p = self._extract(art, lo)
            lbl_p = self._extract(cat, lo)

            if self.train:
                img_p, lbl_p, art_p = self._augment(img_p, lbl_p, extra_p=art_p)

            ch0 = torch.from_numpy(np.ascontiguousarray(img_p)).unsqueeze(0)
            ch1 = torch.from_numpy(np.ascontiguousarray(art_p)).unsqueeze(0)
            img_t = torch.cat([ch0, ch1], dim=0).float()
            lbl_t = torch.from_numpy(np.ascontiguousarray(lbl_p)).unsqueeze(0).float()

        return img_t, lbl_t

## 4. Training utilities

In [ ]:
def dice_loss(pred_logits, target, smooth=1e-5):
    pred = torch.sigmoid(pred_logits)
    inter = (pred * target).sum(dim=(2, 3, 4))
    union = pred.sum(dim=(2, 3, 4)) + target.sum(dim=(2, 3, 4))
    dice = (2 * inter + smooth) / (union + smooth)
    return 1 - dice.mean()


def combo_loss(pred_logits, target, alpha=0.5):
    """0.5 * Dice + 0.5 * BCE."""
    dloss = dice_loss(pred_logits, target)
    bce = F.binary_cross_entropy_with_logits(pred_logits, target, reduction="mean")
    return alpha * dloss + (1 - alpha) * bce


def focal_tversky_loss(pred_logits, target, alpha=0.3, beta=0.7,
                       gamma=0.75, smooth=1e-5):
    """Focal Tversky: Tversky with FN-weighting (beta>alpha) raised to gamma<1
    to focus gradient on hard, partially-segmented examples."""
    p = torch.sigmoid(pred_logits)
    tp = (p * target).sum(dim=(2, 3, 4))
    fp = (p * (1 - target)).sum(dim=(2, 3, 4))
    fn = ((1 - p) * target).sum(dim=(2, 3, 4))
    tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return ((1 - tversky) ** gamma).mean()


def ft_bce_combo(pred_logits, target, w_ft=0.7, w_bce=0.3):
    """Focal-Tversky + BCE — used for Task A from-scratch training."""
    ft = focal_tversky_loss(pred_logits, target)
    bce = F.binary_cross_entropy_with_logits(pred_logits, target)
    return w_ft * ft + w_bce * bce


def catheter_loss(pred_logits, target):
    """Task B loss: same as Task A."""
    return combo_loss(pred_logits, target)

In [ ]:
def plot_history(history: dict, title: str = "Training"):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["train"], label="Train")
    ax.plot(history["val"], label="Val")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()


def train_unet(model: nn.Module,
               train_loader: DataLoader,
               val_loader: DataLoader,
               n_epochs: int,
               loss_fn,
               lr: float = 1e-4,
               ckpt_path: str = "best_model.pt",
               ckpt_every: int = 10,
               grad_clip: float = 1.0,
               use_amp: bool = True):
    """AMP only used on CUDA — falls back gracefully on MPS/CPU."""
    use_amp = use_amp and DEVICE.type == "cuda"
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_val_loss = float("inf")
    history = {"train": [], "val": []}

    ckpt_dir  = Path(ckpt_path).parent
    ckpt_stem = Path(ckpt_path).stem

    for epoch in range(1, n_epochs + 1):
        model.train()
        tloss = 0.0
        for img, lbl in train_loader:
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            opt.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(img)
                loss = loss_fn(pred, lbl)
                if loss.ndim > 0:
                    loss = loss.mean()

            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt)
            scaler.update()
            tloss += loss.item()
        sched.step()
        tloss /= len(train_loader)

        model.eval()
        vloss = 0.0
        with torch.no_grad():
            for img, lbl in val_loader:
                img, lbl = img.to(DEVICE), lbl.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    pred = model(img)
                    loss = loss_fn(pred, lbl)
                    if loss.ndim > 0:
                        loss = loss.mean()
                vloss += loss.item()
        vloss /= len(val_loader)

        history["train"].append(tloss)
        history["val"].append(vloss)

        if vloss < best_val_loss:
            best_val_loss = vloss
            torch.save(model.state_dict(), ckpt_path)

        if ckpt_every and epoch % ckpt_every == 0:
            torch.save(model.state_dict(), str(ckpt_dir / f"{ckpt_stem}_epoch_{epoch}.pt"))

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{n_epochs} | train {tloss:.5f} | val {vloss:.5f}")

    print(f"  Best val loss: {best_val_loss:.5f} — saved to {ckpt_path}")
    return history


def train_unet_dice_ckpt(model, train_loader, val_loader, val_records,
                         n_epochs, loss_fn, lr, ckpt_path,
                         eval_every=3, eval_subjects=6,
                         patch_size=(96, 96, 96), stride=48,
                         grad_clip=1.0, use_amp=True):
    """
    Same training loop as train_unet, but selects checkpoint by max mean
    whole-volume Dice on the first `eval_subjects` records of val_records,
    evaluated every `eval_every` epochs. Per-subject Dice printed each eval.

    No TTA in-loop (too slow); TTA is applied only at final test-time eval.

    Returns history dict with: train_loss[], val_loss[], val_dice_epoch[],
    val_dice_mean[], val_dice_per_subject[].
    """
    use_amp = use_amp and DEVICE.type == "cuda"
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_dice = -1.0
    history = {
        "train_loss": [], "val_loss": [],
        "val_dice_epoch": [], "val_dice_mean": [],
        "val_dice_per_subject": [],
    }
    eval_subs = val_records[:eval_subjects]

    for epoch in range(1, n_epochs + 1):
        model.train()
        tloss = 0.0
        for img, lbl in train_loader:
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(img)
                loss = loss_fn(pred, lbl)
                if loss.ndim > 0:
                    loss = loss.mean()
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt)
            scaler.update()
            tloss += loss.item()
        sched.step()
        tloss /= max(len(train_loader), 1)

        model.eval()
        vloss = 0.0
        with torch.no_grad():
            for img, lbl in val_loader:
                img, lbl = img.to(DEVICE), lbl.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    pred = model(img)
                    loss = loss_fn(pred, lbl)
                    if loss.ndim > 0:
                        loss = loss.mean()
                vloss += loss.item()
        vloss /= max(len(val_loader), 1)

        history["train_loss"].append(tloss)
        history["val_loss"].append(vloss)

        do_eval = (epoch % eval_every == 0) or (epoch == n_epochs)
        if do_eval:
            per_subj = {}
            with torch.no_grad():
                for rec in eval_subs:
                    vol_pre = preprocess_volume(rec["image"], n4=True)
                    gt_art, _, _ = load_nifti(rec["artifact"])
                    pred_bin = sliding_window_inference(
                        model, vol_pre,
                        patch_size=tuple(patch_size), stride=stride)
                    pred_bin = post_process_artifact(pred_bin)
                    inter = float((pred_bin.astype(bool) & gt_art.astype(bool)).sum())
                    denom = float(pred_bin.sum() + gt_art.sum() + 1e-8)
                    per_subj[rec["id"]] = 2.0 * inter / denom

            mean_d = float(np.mean(list(per_subj.values())))
            history["val_dice_epoch"].append(epoch)
            history["val_dice_mean"].append(mean_d)
            history["val_dice_per_subject"].append(per_subj)

            print(f"  Epoch {epoch:3d}/{n_epochs} | train {tloss:.5f} | "
                  f"val {vloss:.5f} | dice {mean_d:.4f}")
            for sid, d in per_subj.items():
                print(f"    {sid}: {d:.4f}")

            if mean_d > best_dice:
                best_dice = mean_d
                torch.save(model.state_dict(), ckpt_path)
                print(f"    ↳ saved (best mean Dice {best_dice:.4f})")
        else:
            if epoch % 5 == 0 or epoch == 1:
                print(f"  Epoch {epoch:3d}/{n_epochs} | train {tloss:.5f} | "
                      f"val {vloss:.5f}")

    print(f"  Best mean val Dice: {best_dice:.4f} — saved to {ckpt_path}")
    return history

## 5. Inference utilities

In [11]:
def sliding_window_inference(model: nn.Module,
                             vol: np.ndarray,
                             patch_size: Tuple = (64, 64, 64),
                             stride: int = 32,
                             extra_ch: Optional[np.ndarray] = None,
                             threshold: float = 0.5,
                             return_prob: bool = False) -> np.ndarray:
    """Gaussian-weighted sliding window over full volume."""
    model.eval()
    ps = np.array(patch_size)
    sh = np.array(vol.shape)
    prob = np.zeros(sh, dtype=np.float32)
    cnt  = np.zeros(sh, dtype=np.float32)

    g1d  = np.exp(-4 * (np.linspace(-1, 1, ps[0]) ** 2))
    kern = g1d[:, None, None] * g1d[None, :, None] * g1d[None, None, :]

    starts = [range(0, max(s - p, 1), stride) for s, p in zip(sh, ps)]
    for xs, ys, zs in itertools.product(*starts):
        xe = min(xs + ps[0], sh[0]); ye = min(ys + ps[1], sh[1]); ze = min(zs + ps[2], sh[2])
        xs2, ys2, zs2 = xe - ps[0], ye - ps[1], ze - ps[2]

        patch  = vol[xs2:xe, ys2:ye, zs2:ze]
        tensor = torch.from_numpy(patch[None, None]).float().to(DEVICE)
        if extra_ch is not None:
            ext = extra_ch[xs2:xe, ys2:ye, zs2:ze]
            tensor = torch.cat([tensor,
                                torch.from_numpy(ext[None, None]).float().to(DEVICE)], dim=1)
        with torch.no_grad():
            logit = model(tensor)[0, 0].cpu().numpy()
        pred_prob = 1 / (1 + np.exp(-logit))

        prob[xs2:xe, ys2:ye, zs2:ze] += pred_prob * kern
        cnt [xs2:xe, ys2:ye, zs2:ze] += kern

    prob /= (cnt + 1e-8)
    if return_prob:
        return prob
    return (prob >= threshold).astype(np.uint8)


def sliding_window_inference_tta(model: nn.Module,
                                 vol: np.ndarray,
                                 patch_size: Tuple = (64, 64, 64),
                                 stride: int = 32,
                                 extra_ch: Optional[np.ndarray] = None,
                                 threshold: float = 0.5) -> np.ndarray:
    """
    8-flip TTA. Un-flips each probability map BEFORE averaging.
    Both vol and extra_ch are flipped consistently.
    """
    prob_sum = np.zeros(vol.shape, dtype=np.float32)
    for fz, fy, fx in itertools.product([False, True], repeat=3):
        axes = tuple(a for a, f in zip((0, 1, 2), (fz, fy, fx)) if f)

        vol_f   = np.flip(vol, axis=axes).copy() if axes else vol
        extra_f = None
        if extra_ch is not None:
            extra_f = np.flip(extra_ch, axis=axes).copy() if axes else extra_ch

        prob_f = sliding_window_inference(model, vol_f,
                                          patch_size=patch_size, stride=stride,
                                          extra_ch=extra_f, return_prob=True)
        prob_sum += np.flip(prob_f, axis=axes).copy() if axes else prob_f

    return ((prob_sum / 8.0) >= threshold).astype(np.uint8)


def post_process_artifact(mask: np.ndarray) -> np.ndarray:
    """Keep only the largest connected component (one valve per scan)."""
    labeled_arr, n = label(mask)
    if n == 0:
        return mask
    sizes = [np.sum(labeled_arr == i) for i in range(1, n + 1)]
    biggest = np.argmax(sizes) + 1
    return (labeled_arr == biggest).astype(np.uint8)


def post_process_catheter(cath_mask: np.ndarray,
                          art_mask: np.ndarray,
                          close_radius: int = 1,
                          min_cc_fraction: float = 0.1) -> np.ndarray:
    """Zero overlap with artifact, morph close, keep large CCs, re-zero overlap."""
    cath = np.where(art_mask > 0, 0, cath_mask).astype(np.uint8)
    if cath.sum() == 0:
        return cath

    if close_radius > 0:
        struct = ball(close_radius)
        cath = binary_dilation(cath.astype(bool), structure=struct)
        cath = binary_erosion(cath, structure=struct).astype(np.uint8)
        cath = np.where(art_mask > 0, 0, cath).astype(np.uint8)

    labeled_arr, n = label(cath)
    if n == 0:
        return cath
    sizes = np.array([np.sum(labeled_arr == i) for i in range(1, n + 1)])
    biggest = sizes.max()
    keep_ids = (np.where(sizes >= max(1, min_cc_fraction * biggest))[0] + 1).tolist()
    return np.isin(labeled_arr, keep_ids).astype(np.uint8)


def infer_artifact(model_a, rec, patch_size, stride=32):
    """Sliding-window Task A inference."""
    vol = preprocess_volume(rec["image"])
    return sliding_window_inference(model_a, vol, patch_size=patch_size, stride=stride)

## 6. Length estimation (Task C)

In [12]:
def prune_skeleton(skeleton):
    """Remove branch points, keep largest connected component of what remains."""
    kernel = np.ones((3, 3, 3), dtype=int)
    kernel[1, 1, 1] = 0
    neighbor_count = convolve(skeleton.astype(int), kernel, mode='constant')
    pruned = skeleton.copy()
    pruned[(pruned > 0) & (neighbor_count > 2)] = 0
    labeled, n = label(pruned)
    if n == 0:
        return skeleton  # fallback if pruning removed everything
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    return (labeled == sizes.argmax()).astype(skeleton.dtype)


def extend_endpoint_vox(endpoint_vox, inner_vox, mask, max_steps=20):
    """Walk outward from endpoint along catheter direction until mask ends."""
    direction = np.array(endpoint_vox, dtype=float) - np.array(inner_vox, dtype=float)
    norm = np.linalg.norm(direction)
    if norm < 1e-9:
        return np.array(endpoint_vox)
    direction /= norm
    p = np.array(endpoint_vox, dtype=float)
    last_valid = np.array(endpoint_vox)
    for _ in range(max_steps):
        p += direction
        pi = np.round(p).astype(int)
        if not all(0 <= pi[i] < mask.shape[i] for i in range(3)):
            break
        if mask[pi[0], pi[1], pi[2]] == 0:
            break
        last_valid = pi.copy()
    return last_valid


def extract_ordered_skeleton(mask: np.ndarray, spacing: np.ndarray) -> np.ndarray:
    struct = ball(1)
    dilated = binary_dilation(mask.astype(bool), struct)
    skel = skeletonize(dilated)
    skel = prune_skeleton(skel)
    coords_vox = np.argwhere(skel)

    if len(coords_vox) < 2:
        coords_vox = np.argwhere(mask > 0)
        if len(coords_vox) < 2:
            return coords_vox * spacing[None, :]

    coords_mm = coords_vox * spacing[None, :]

    # 2-pass endpoint finding
    seed = coords_mm[0]
    dists = np.linalg.norm(coords_mm - seed, axis=1)
    a_idx = np.argmax(dists)
    dists_a = np.linalg.norm(coords_mm - coords_mm[a_idx], axis=1)

    # Greedy nearest-neighbor walk from endpoint A
    visited = [a_idx]
    remaining = set(range(len(coords_mm))) - {a_idx}
    while remaining:
        last = visited[-1]
        cands = list(remaining)
        dists = np.linalg.norm(coords_mm[cands] - coords_mm[last], axis=1)
        nxt = cands[np.argmin(dists)]
        visited.append(nxt)
        remaining.remove(nxt)

    ordered_vox = coords_vox[visited]

    # Extend both endpoints outward to the mask boundary
    if len(ordered_vox) >= 2:
        ext_start = extend_endpoint_vox(ordered_vox[0],  ordered_vox[1],  mask)
        ext_end   = extend_endpoint_vox(ordered_vox[-1], ordered_vox[-2], mask)
        ordered_vox = np.vstack([ext_start[None, :], ordered_vox, ext_end[None, :]])

    return ordered_vox * spacing[None, :]


def fit_best_spline(ordered_pts_mm: np.ndarray, mask: np.ndarray,
                    spacing: np.ndarray, max_anchor: int = 10):
    mask_coords = np.argwhere(mask > 0).astype(float) * spacing[None, :]
    best_mse = np.inf
    best_length = 0.0
    best_tck = None

    for anchor_step in range(1, max_anchor + 1):
        anchors = ordered_pts_mm[::anchor_step]
        if not np.allclose(anchors[-1], ordered_pts_mm[-1]):
            anchors = np.vstack([anchors, ordered_pts_mm[-1]])
        if len(anchors) < 4:
            continue
        try:
            tck, u = splprep(anchors.T, s=0, k=3)
        except Exception:
            continue

        t_fine = np.linspace(0, 1, 1000)
        dx, dy, dz = splev(t_fine, tck, der=1)
        speed = np.sqrt(dx**2 + dy**2 + dz**2)
        length = float(np.trapz(speed, t_fine))

        x, y, z = splev(t_fine, tck)
        spline_pts = np.column_stack([x, y, z])
        tree = cKDTree(spline_pts)
        dists, _ = tree.query(mask_coords)
        mse = np.mean(dists**2)

        if mse < best_mse:
            best_mse = mse
            best_length = length
            best_tck = tck

    if best_tck is None:
        diffs = np.diff(ordered_pts_mm, axis=0)
        best_length = float(np.linalg.norm(diffs, axis=1).sum())

    return best_length, best_tck


def bridge_artifact_gap(cath_mask: np.ndarray, art_mask: np.ndarray,
                        spacing: np.ndarray) -> float:
    """Estimate hidden catheter length inside the artifact."""
    art_dilated = binary_dilation(art_mask.astype(bool), ball(2))
    interface = art_dilated & cath_mask.astype(bool)
    if interface.sum() == 0:
        return 0.0
    cath_tip_mm = np.argwhere(interface).mean(axis=0) * spacing
    art_pts = np.argwhere(art_mask > 0)
    if len(art_pts) == 0:
        return 0.0
    valve_mm = art_pts.mean(axis=0) * spacing
    return float(np.linalg.norm(valve_mm - cath_tip_mm))


def estimate_catheter_length(cath_mask: np.ndarray, art_mask: np.ndarray,
                             spacing: np.ndarray) -> float:
    """Task C: visible spline length + artifact gap = total length in mm."""
    if cath_mask.sum() == 0:
        return 0.0
    ordered_pts = extract_ordered_skeleton(cath_mask, spacing)
    if len(ordered_pts) < 2:
        return 0.0
    visible_len, _ = fit_best_spline(ordered_pts, cath_mask, spacing)
    gap_len = bridge_artifact_gap(cath_mask, art_mask, spacing)
    return visible_len + gap_len

## 7. Evaluation utilities

In [ ]:
def dsc(pred, gt):
    """Dice similarity coefficient."""
    p, g = pred.astype(bool), gt.astype(bool)
    return 2 * (p & g).sum() / (p.sum() + g.sum() + 1e-8)


def evaluate_task_a(model_a, val_records):
    """Per-subject Task A Dice over a list of validation records.

    Uses N4 bias-field correction in preprocessing and 8-flip TTA at inference
    for the strongest, most reproducible Task-A score we report.
    """
    model_a.eval()
    scores = {}
    with torch.no_grad():
        for rec in val_records:
            sid = rec["id"]
            gt_art, _, _ = load_nifti(rec["artifact"])
            vol_pre = preprocess_volume(rec["image"], n4=True)
            pred_bin = sliding_window_inference_tta(
                model_a, vol_pre,
                patch_size=tuple(CFG["patch_size"]), stride=32)
            pred_bin = post_process_artifact(pred_bin)
            d = dsc(pred_bin, gt_art)
            scores[sid] = d
            print(f"{sid}: {d:.4f}")
    print(f"\nMean Task A Dice: {np.mean(list(scores.values())):.4f}")
    return scores


def evaluate_val_set(val_records, model_a, model_b, max_subjects=5):
    """Full pipeline eval: A Dice, B Dice, C RAE on a subset of val subjects."""
    model_a.eval(); model_b.eval()
    rows = []

    for rec in tqdm(val_records[:max_subjects], desc="Evaluating"):
        vol, sp, _ = load_nifti(rec["image"])
        vol_pre = preprocess_volume(rec["image"])
        gt_art, _, _ = load_nifti(rec["artifact"])
        gt_cat, _, _ = load_nifti(rec["catheter"])

        pred_art = post_process_artifact(
            sliding_window_inference(model_a, vol_pre,
                                     patch_size=CFG["patch_size"], stride=32))
        pred_cat = post_process_catheter(
            sliding_window_inference(model_b, vol_pre,
                                     patch_size=CFG["patch_size"], stride=32,
                                     extra_ch=pred_art.astype(np.float32)),
            pred_art)

        dsc_a = dsc(pred_art, gt_art)
        dsc_b = dsc(pred_cat, gt_cat)

        pred_len = estimate_catheter_length(pred_cat, pred_art, sp)
        gt_len = None
        if "json" in rec and rec["json"].exists():
            with open(rec["json"]) as f:
                gt_len = json.load(f)["catheter_length_mm"]
        rae = abs(pred_len - gt_len) / (gt_len + 1e-8) if gt_len else None

        rows.append(dict(subject=rec["id"], dsc_artifact=dsc_a,
                         dsc_catheter=dsc_b, rae_length=rae,
                         pred_len_mm=pred_len, gt_len_mm=gt_len))

    df = pd.DataFrame(rows)
    print(f"\nDSC Artifact  (mean ± std): {df.dsc_artifact.mean():.4f} ± {df.dsc_artifact.std():.4f}")
    print(f"DSC Catheter  (mean ± std): {df.dsc_catheter.mean():.4f} ± {df.dsc_catheter.std():.4f}")
    if df.rae_length.notna().any():
        print(f"RAE Length    (median):     {df.rae_length.median():.4f}")
    return df


def visualise_subject(rec, model_a, model_b, axial_z=None):
    """6-panel comparison: GT vs predicted masks overlaid on MRI."""
    vol, sp, _ = load_nifti(rec["image"])
    vol_pre = preprocess_volume(rec["image"])

    pred_art = post_process_artifact(
        sliding_window_inference(model_a, vol_pre, patch_size=CFG["patch_size"], stride=48))
    pred_cat = post_process_catheter(
        sliding_window_inference(model_b, vol_pre, patch_size=CFG["patch_size"], stride=48,
                                 extra_ch=pred_art.astype(np.float32)),
        pred_art)

    gt_art, _, _ = load_nifti(rec["artifact"])
    gt_cat, _, _ = load_nifti(rec["catheter"])

    z = axial_z if axial_z else vol.shape[2] // 2
    mri_sl = vol[:, :, z].T

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    titles = [["MRI", "GT Artifact", "GT Catheter"],
              ["MRI", "Pred Artifact", "Pred Catheter"]]
    for r in range(2):
        for c in range(3):
            axes[r, c].set_title(titles[r][c])
            axes[r, c].axis("off")

    axes[0, 0].imshow(mri_sl, cmap="gray", origin="lower")
    axes[0, 1].imshow(mri_sl, cmap="gray", origin="lower")
    axes[0, 1].imshow(gt_art[:, :, z].T, cmap="Reds", origin="lower", alpha=0.5)
    axes[0, 2].imshow(mri_sl, cmap="gray", origin="lower")
    axes[0, 2].imshow(gt_cat[:, :, z].T, cmap="Blues", origin="lower", alpha=0.5)
    axes[1, 0].imshow(mri_sl, cmap="gray", origin="lower")
    axes[1, 1].imshow(mri_sl, cmap="gray", origin="lower")
    axes[1, 1].imshow(pred_art[:, :, z].T, cmap="Reds", origin="lower", alpha=0.5)
    axes[1, 2].imshow(mri_sl, cmap="gray", origin="lower")
    axes[1, 2].imshow(pred_cat[:, :, z].T, cmap="Blues", origin="lower", alpha=0.5)

    plt.suptitle(f"{rec['id']} | z={z}", fontsize=13)
    plt.tight_layout(); plt.show()

## 8. Train/val split

In [14]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")

records = get_subject_paths(DATA_ROOT)
n_val = int(len(records) * CFG["val_frac"])
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(records))
train_r = [records[i] for i in idx[n_val:]]
val_r   = [records[i] for i in idx[:n_val]]
print(f"Train: {len(train_r)} subjects | Val: {len(val_r)} subjects")

FileNotFoundError: DATA_ROOT not found: /content/drive/MyDrive/Colab Notebooks/MIAProject2/data/Train

## 9. Task A — Artifact segmentation

Toggle Option A (load checkpoint) vs Option B (fine-tune). Default: load.

In [ ]:
# ── OPTION A: Load from checkpoint (default) ─────────────────────────────────
# Prefer the new v2 checkpoint; fall back to model_a_best.pt so the previous
# fine-tune isn't overwritten by accident. Skips gracefully on a fresh setup
# so Option B can still run when no checkpoints exist yet.
v2_path       = OUT_DIR / "model_a_v2.pt"
fallback_path = OUT_DIR / "model_a_best.pt"
ckpt_to_load  = v2_path if v2_path.exists() else (fallback_path if fallback_path.exists() else None)

model_a = UNet3D(in_channels=1, base=16).to(DEVICE)
if ckpt_to_load is not None:
    model_a.load_state_dict(torch.load(str(ckpt_to_load), map_location=DEVICE))
    print(f"Loaded model_a from {ckpt_to_load.name}.")
else:
    print("No Task A checkpoint found — initialised model_a from scratch.")


# ── OPTION B: Train Task A from scratch with v2 pipeline ─────────────────────
# Pipeline: TorchIO aug + N4 (eval) + 96³ patches + Focal-Tversky+BCE +
#           whole-volume Dice checkpointing on first 6 val subjects.
# Saves to model_a_v2.pt — leaves model_a_best.pt untouched.
ds_a_tr  = PatchDataset(train_r, task='A', train=True,
                        patch_size=CFG["patch_size"],
                        patches_per_vol=16, fg_oversample=0.5)
ds_a_val = PatchDataset(val_r, task='A', train=False,
                        patch_size=CFG["patch_size"],
                        patches_per_vol=4, fg_oversample=0.5)
dl_a_tr  = DataLoader(ds_a_tr, batch_size=CFG["batch_size"],
                      shuffle=True, num_workers=CFG["n_workers"], pin_memory=True)
dl_a_val = DataLoader(ds_a_val, batch_size=CFG["batch_size"],
                      shuffle=False, num_workers=CFG["n_workers"], pin_memory=True)

model_a = UNet3D(in_channels=1, base=16).to(DEVICE)   # train from scratch
hist_a = train_unet_dice_ckpt(
    model_a, dl_a_tr, dl_a_val,
    val_records=val_r,
    n_epochs=CFG["epochs_a"],
    loss_fn=ft_bce_combo,
    lr=3e-4,
    ckpt_path=str(OUT_DIR / "model_a_v2.pt"),
    eval_every=3,
    eval_subjects=6,
    patch_size=CFG["patch_size"],
    stride=48,
)

# ─── Plot patch losses + sparse whole-volume Dice ────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(range(1, len(hist_a["train_loss"]) + 1), hist_a["train_loss"],
         label="Train", color="steelblue")
ax1.plot(range(1, len(hist_a["val_loss"]) + 1), hist_a["val_loss"],
         label="Val (patch)", color="orange")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Patch loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
# mean Dice line
ax2.plot(hist_a["val_dice_epoch"], hist_a["val_dice_mean"],
         marker="o", color="green", label="Mean val Dice")
# per-subject scatter
for ep, ps in zip(hist_a["val_dice_epoch"], hist_a["val_dice_per_subject"]):
    ax2.scatter([ep] * len(ps), list(ps.values()),
                color="green", alpha=0.3, s=20)
ax2.set_ylabel("Whole-volume Dice"); ax2.set_ylim(0, 1)
ax2.legend(loc="lower right")
plt.title("Task A v2 — TorchIO + N4 + 96³ + Focal-Tversky+BCE")
plt.tight_layout(); plt.show()


# ─── TODO (out of scope for this iteration) ──────────────────────────────────
# - 5-fold cross-validation / ensembling
# - base=32 or deeper U-Net
# - Cascaded localization → segmentation

In [ ]:
# Task A per-subject Dice on val set (uses N4 + TTA inside evaluate_task_a)
task_a_scores = evaluate_task_a(model_a, val_r)

In [ ]:
# ─── Failure-mode diagnostic: Dice vs artifact size + Dice histogram ─────────
# Verifies that the bimodal split (many ≥0.90, many ≤0.60) is closing.
import matplotlib as mpl

PINK      = "#F5B7B1"   # pastel pink fill
PINK_DARK = "#922B21"   # accent for annotations

# Artifact voxel counts (GT) for the val set
gt_voxels = {}
for rec in val_r:
    gt_art, _, _ = load_nifti(rec["artifact"])
    gt_voxels[rec["id"]] = int(gt_art.sum())

sids  = list(task_a_scores.keys())
dices = np.array([task_a_scores[s] for s in sids], dtype=float)
sizes = np.array([gt_voxels[s]    for s in sids], dtype=float)

worst3 = np.argsort(dices)[:3]
mean_d = float(dices.mean())
n_below = int((dices < 0.6).sum())

style = {
    "font.family":         "Arial",
    "font.weight":         "bold",
    "axes.labelweight":    "bold",
    "axes.titleweight":    "bold",
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.spines.left":    True,
    "axes.spines.bottom":  True,
}

with mpl.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # ── Scatter ─────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.scatter(sizes, dices, color=PINK, edgecolor="black",
               linewidth=0.8, s=70, zorder=3)
    ax.axhline(0.6, color="black", linestyle="--", linewidth=1, zorder=2)
    for i in worst3:
        ax.annotate(sids[i], (sizes[i], dices[i]),
                    xytext=(6, 6), textcoords="offset points",
                    fontsize=9, fontweight="bold", color=PINK_DARK)
    ax.set_xlabel("Artifact voxel count (GT)")
    ax.set_ylabel("Per-subject Dice")
    ax.set_title("Dice vs artifact size")
    ax.set_ylim(-0.02, 1.02)

    # ── Histogram ───────────────────────────────────────────────────────────
    ax = axes[1]
    bins = np.arange(0.0, 1.05, 0.05)
    ax.hist(dices, bins=bins, color=PINK, edgecolor="black",
            linewidth=0.8, rwidth=0.75)
    ax.set_xlabel("Per-subject Dice")
    ax.set_ylabel("Count")
    ax.set_title("Val Dice distribution")
    ax.set_xlim(0.0, 1.0)

    plt.suptitle(f"Mean Dice {mean_d:.3f}   |   "
                 f"{n_below}/{len(dices)} subjects below 0.60",
                 fontweight="bold")
    plt.tight_layout(); plt.show()

## 10. Task B — Catheter segmentation

Toggle Option A (load checkpoint) vs Option B (fine-tune). Default: load.

In [16]:
import shutil
shutil.rmtree(str(CACHE_DIR))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Cache cleared.")

Cache cleared.


In [ ]:
# ── OPTION A: Load from checkpoint ────────────────────────────────────────────
# model_b = UNet3D(in_channels=2, base=16).to(DEVICE)
# model_b.load_state_dict(torch.load(str(OUT_DIR / "model_b_best.pt"), map_location=DEVICE))
# print("Loaded model_b from checkpoint.")

# ── OPTION B: Fine-tune Task B from current checkpoint ────────────────────────
ds_b_tr = PatchDataset(train_r, task='B', patch_size=CFG["patch_size"],
                       patches_per_vol=16, fg_oversample=0.7)
ds_b_val = PatchDataset(val_r, task='B', patch_size=CFG["patch_size"],
                        patches_per_vol=4, fg_oversample=0.7)
dl_b_tr = DataLoader(ds_b_tr, batch_size=CFG["batch_size"],
                     shuffle=True, num_workers=CFG["n_workers"], pin_memory=True)
dl_b_val = DataLoader(ds_b_val, batch_size=CFG["batch_size"],
                      shuffle=False, num_workers=CFG["n_workers"], pin_memory=True)
model_b = UNet3D(in_channels=2, base=16).to(DEVICE)
model_b.load_state_dict(torch.load(str(OUT_DIR / "model_b_best.pt"), map_location=DEVICE))
hist_b = train_unet(model_b, dl_b_tr, dl_b_val,
                    n_epochs=CFG["epochs_b"],
                    loss_fn=catheter_loss,
                    lr=1e-4,
                    ckpt_path=str(OUT_DIR / "model_b_best.pt"))
plot_history(hist_b, title="Task B — fine-tune")

  Epoch   1/60 | train 0.11697 | val 0.14907


In [ ]:
# Full-pipeline eval (A Dice, B Dice, C RAE) on a subset of val subjects
val_df = evaluate_val_set(val_r, model_a, model_b, max_subjects=5)
val_df.to_csv(str(OUT_DIR / "validation_results.csv"), index=False)
display(val_df)

## 11. Visualization

In [ ]:
# Pick the slice with most artifact content for the first val subject
rec = val_r[0]
gt_art, _, _ = load_nifti(rec["artifact"])
z_best = int(np.argmax(gt_art.sum(axis=(0,1))))
print(f"Best slice: z={z_best}")
visualise_subject(rec, model_a, model_b, axial_z=z_best)

## 12. Test inference & submission

Generates `subjXXX_taskA.nii.gz`, `subjXXX_taskB.nii.gz`, and `taskC.csv` in `OUT_DIR`.

In [ ]:
def run_test_inference(test_root, model_a, model_b, out_dir=OUT_DIR):
    out_dir.mkdir(exist_ok=True)
    test_records = get_subject_paths(test_root, labeled=False)
    length_rows = []

    model_a.eval(); model_b.eval()

    for rec in tqdm(test_records, desc="Test inference"):
        vol, sp, ref_img = load_nifti(rec["image"])
        vol_pre = preprocess_volume(rec["image"])
        sid = rec["id"]

        pred_art = post_process_artifact(
            infer_artifact(model_a, rec,
                           patch_size=CFG["patch_size"], stride=32))
        save_nifti(pred_art, ref_img, out_dir / f"{sid}_taskA.nii.gz")

        pred_cat = post_process_catheter(
            sliding_window_inference(model_b, vol_pre,
                                     patch_size=CFG["patch_size"], stride=32,
                                     extra_ch=pred_art.astype(np.float32)),
            pred_art)
        save_nifti(pred_cat, ref_img, out_dir / f"{sid}_taskB.nii.gz")

        length = estimate_catheter_length(pred_cat, pred_art, sp)
        length_rows.append({"subject": sid, "length": round(length, 2)})

    df_c = pd.DataFrame(length_rows)
    df_c.to_csv(out_dir / "taskC.csv", index=False)
    print(f"Saved {len(test_records)} predictions to {out_dir}/")
    return df_c


if TEST_ROOT.exists():
    df_lengths = run_test_inference(TEST_ROOT, model_a, model_b)
    print(df_lengths.head())
else:
    print(f"Test data not found at {TEST_ROOT} — add it to run submission inference.")

## 13. Sanity checks

GT-only Task C eval (no U-Net needed) and the official grading script.

In [ ]:
def evaluate_length_on_gt(data_root: Path):
    """Test the spline-fitting pipeline in isolation using GT masks."""
    records = get_subject_paths(data_root)
    errors = []
    for rec in tqdm(records, desc="GT length eval"):
        cat, sp, _ = load_nifti(rec["catheter"])
        art, _, _ = load_nifti(rec["artifact"])
        pred_len = estimate_catheter_length(
            cat.astype(np.uint8), art.astype(np.uint8), sp)
        if "json" in rec and rec["json"].exists():
            with open(rec["json"]) as f:
                gt_len = json.load(f)["catheter_length_mm"]
            errors.append(pred_len - gt_len)

    errors = np.array(errors)
    print(f"Mean Absolute Error: {np.mean(np.abs(errors)):.2f} mm")
    print(f"Mean Error (bias):   {np.mean(errors):.2f} mm")
    print(f"Std Error:           {np.std(errors):.2f} mm")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(errors, bins=np.arange(min(errors), max(errors) + 0.5, 0.5),
            color="steelblue", edgecolor="white")
    ax.axvline(np.mean(errors), color="red", linestyle="--",
               label=f"Mean: {np.mean(errors):.2f}")
    ax.set_xlabel("Error (mm)"); ax.set_ylabel("Count")
    ax.set_title("Length Estimation Error on GT Masks")
    ax.legend(); plt.tight_layout(); plt.show()
    return errors


# Uncomment to run (~10-20 min on 200 subjects):
# gt_errors = evaluate_length_on_gt(DATA_ROOT)

In [ ]:
# Run the official grading script against training GT
eval_script = PROJECT_ROOT / "data" / "evaluate.py"
if eval_script.exists() and OUT_DIR.exists():
    !python "{eval_script}" \
        --gt-dir "{DATA_ROOT}" \
        --pred-dir "{OUT_DIR}" \
        --output-csv "{OUT_DIR / 'eval_results.csv'}"
else:
    print("Grading script not found — skipping.")